In [1]:
import pandas as pd

# Si has subido los archivos manualmente a Colab, estarán en /content/
train_path = "/content/train.csv"
test_path  = "/content/test.csv"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print("train:", train.shape)
print("test :", test.shape)

train.head()


train: (26570, 26)
test : (20775, 25)


,id,product_code,loading,attribute_0,attribute_1,attribute_2,attribute_3,measurement_0,measurement_1,measurement_2,...,measurement_9,measurement_10,measurement_11,measurement_12,measurement_13,measurement_14,measurement_15,measurement_16,measurement_17,failure
0,0,A,80.10,material_7,material_8,9,5,7,8,4,...,10.672,15.859,17.594,15.193,15.029,NaN,13.034,14.684,764.100,0
1,1,A,84.89,material_7,material_8,9,5,14,3,3,...,12.448,17.947,17.915,11.755,14.732,15.425,14.395,15.631,682.057,0
2,2,A,82.43,material_7,material_8,9,5,12,1,5,...,12.715,15.607,NaN,13.798,16.711,18.631,14.094,17.946,663.376,0
3,3,A,101.07,material_7,material_8,9,5,13,2,6,...,12.471,16.346,18.377,10.020,15.250,15.562,16.154,17.172,826.282,0
4,4,A,188.06,material_7,material_8,9,5,9,2,8,...,10.337,17.082,19.932,12.428,16.182,12.760,13.153,16.412,579.885,0


In [2]:
X_train = train.drop(columns=["failure", "id"])
y_train = train["failure"]

X_test = test.drop(columns=["id"])

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape)


X_train: (26570, 24) y_train: (26570,)
X_test : (20775, 24)


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# --- X/y ---
X = train.drop(columns=["failure", "id"])
y = train["failure"]

X_test = test.drop(columns=["id"])

# Detectar tipos de columnas
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical cols:", len(cat_cols))
print("Numeric cols     :", len(num_cols))

# Pipelines de preprocesado
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ]
)

# Modelo final (con pipeline)
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])

# Split y validación
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_tr, y_tr)

val_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_pred)
print("Validation AUC:", auc)


Categorical cols: 3
Numeric cols     : 21
Validation AUC: 0.5456292619164452


In [6]:
model.fit(X, y)

test_pred = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "failure_probability": test_pred
})

submission.to_csv("/content/submission.csv", index=False)
submission.head()


,id,failure_probability
0,26570,0.2325
1,26571,0.1950
2,26572,0.2025
3,26573,0.2175
4,26574,0.3775


In [7]:
from google.colab import files
files.download("/content/submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from sklearn.metrics import roc_auc_score

train_drift = train.drop(columns=["failure", "id"]).copy()
test_drift  = test.drop(columns=["id"]).copy()

train_drift["is_test"] = 0
test_drift["is_test"]  = 1

drift_df = pd.concat([train_drift, test_drift], axis=0).reset_index(drop=True)

X_drift = drift_df.drop(columns=["is_test"])
y_drift = drift_df["is_test"]

# columnas categóricas/numéricas para drift (puede coincidir, pero mejor recalcular)
cat_cols_d = X_drift.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols_d = [c for c in X_drift.columns if c not in cat_cols_d]

preprocess_drift = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols_d),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols_d),
    ]
)

drift_model = Pipeline(steps=[
    ("preprocess", preprocess_drift),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])

Xd_tr, Xd_val, yd_tr, yd_val = train_test_split(
    X_drift, y_drift, test_size=0.2, random_state=42, stratify=y_drift
)

drift_model.fit(Xd_tr, yd_tr)
drift_pred = drift_model.predict_proba(Xd_val)[:, 1]
drift_auc = roc_auc_score(yd_val, drift_pred)

print("Drift detection AUC:", drift_auc)


Drift detection AUC: 1.0


## 🧠 Metodología y Procesamiento de los Datos

Este trabajo se divide en dos tareas principales: (1) la predicción de fallos y (2) la detección de *data drift*. En ambas tareas se ha seguido un pipeline de aprendizaje automático estructurado para garantizar coherencia en el tratamiento de los datos y en la evaluación de los modelos.

---

### 1. Limpieza y Preprocesamiento de los Datos

El conjunto de datos contiene tanto variables numéricas como categóricas. Antes de entrenar los modelos, se realizaron los siguientes pasos de preprocesamiento:

- Se eliminó la columna `id`, ya que actúa únicamente como identificador y no aporta información predictiva.
- En el conjunto de entrenamiento, la variable `failure` se separó como variable objetivo.
- Los valores faltantes se trataron mediante imputación:
  - Las variables numéricas se imputaron utilizando la mediana.
  - Las variables categóricas se imputaron con el valor más frecuente.
- Las variables categóricas se transformaron mediante *One-Hot Encoding* para convertirlas en una representación numérica adecuada para los modelos de aprendizaje automático.

Todo el preprocesamiento se implementó mediante un `Pipeline` y un `ColumnTransformer`, asegurando que las mismas transformaciones se aplicaran de forma consistente tanto a los datos de entrenamiento como a los de test, evitando así fugas de información (*data leakage*).

---

### 2. Modelo de Predicción de Fallos (Task 1)

Para la tarea de predicción de fallos se utilizó un **clasificador Random Forest**, elegido por su robustez y su buen rendimiento en conjuntos de datos tabulares con variables heterogéneas.

El conjunto de entrenamiento se dividió en dos subconjuntos, utilizando un 80 % de los datos para entrenamiento y un 20 % para validación. El rendimiento del modelo se evaluó mediante la métrica **AUC (Area Under the ROC Curve)**, que mide la capacidad del modelo para discriminar entre productos que fallan y productos que no fallan.

Tras la validación, el modelo se reentrenó utilizando la totalidad del conjunto de entrenamiento y se empleó para generar predicciones de probabilidad de fallo sobre el conjunto de test. Estas predicciones se almacenaron en un archivo de salida para su posterior evaluación.

---

### 3. Detección de Data Drift (Task 2)

En la segunda tarea se analizó la presencia de *data drift* reformulando el problema como una tarea de clasificación binaria. El objetivo fue determinar si una observación provenía del conjunto de entrenamiento o del conjunto de test.

Para ello:
- Se combinaron los conjuntos de entrenamiento y test.
- Se eliminaron las columnas `failure` e `id`.
- Se creó una nueva variable binaria `is_test`, donde:
  - `0` indica observaciones del conjunto de entrenamiento.
  - `1` indica observaciones del conjunto de test.

Se aplicó el mismo pipeline de preprocesamiento utilizado en la Task 1 para mantener la coherencia entre ambas tareas.

A continuación, se entrenó un modelo Random Forest para distinguir entre datos de entrenamiento y de test, evaluando nuevamente el rendimiento mediante la métrica AUC.

Un valor de **AUC = 1.0** indica que el modelo es capaz de distinguir perfectamente entre ambos conjuntos, lo que evidencia la existencia de un *data drift* severo y un cambio significativo en la distribución de los datos.

---

### 4. Interpretación de los Resultados

Los resultados obtenidos muestran que, aunque es posible entrenar un modelo predictivo a partir de datos históricos, su fiabilidad depende en gran medida de la estabilidad de la distribución de los datos. La presencia de un *data drift* elevado sugiere que el modelo de predicción de fallos puede no generalizar correctamente a datos nuevos y que sería necesario reentrenarlo o adaptarlo para mantener su rendimiento.
